In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
train_data = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
train_data.head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
test_data = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [4]:
train_data["FamilySize"] = train_data["SibSp"] + train_data["Parch"] + 1
test_data["FamilySize"] = test_data["SibSp"] + test_data["Parch"] + 1

In [5]:
women = train_data.loc[train_data.Sex == 'female']["Survived"]
rate_women = sum(women)/len(women)

print("% of women who survived:", rate_women)

% of women who survived: 0.7420382165605095


In [6]:
men = train_data.loc[train_data.Sex == 'male']["Survived"]
rate_men = sum(men)/len(men)

print("% of men who survived:", rate_men)

% of men who survived: 0.18890814558058924


In [7]:
train_data.head(50)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FamilySize
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,2
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,2
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,2
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,1
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q,1
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S,1
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S,5
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S,3
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C,2


In [8]:
# Age groups
def age_group(age):
    if pd.isnull(age):
        return "Unknown"
    elif age < 12:
        return "Child"
    elif age < 60:
        return "Adult"
    else:
        return "Senior"

train_data["AgeGroup"] = train_data["Age"].apply(age_group)
test_data["AgeGroup"] = test_data["Age"].apply(age_group)


# Family size
train_data["FamilySize"] = train_data["SibSp"] + train_data["Parch"] + 1
test_data["FamilySize"] = test_data["SibSp"] + test_data["Parch"] + 1

In [9]:
features = [
    "Pclass",
    "Sex",
    "AgeGroup",
    "FamilySize",
    "Fare",
    "Embarked"
]

In [10]:
X = pd.get_dummies(train_data[features])
X_test = pd.get_dummies(test_data[features])

X, X_test = X.align(X_test, join="left", axis=1, fill_value=0)

In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

y = train_data["Survived"]



X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1)

model.fit(X_train, y_train)

# validation predictions
val_preds = model.predict(X_val)

acc = accuracy_score(y_val, val_preds)
print("Validation accuracy:", acc)

# test predictions (for Kaggle)
test_preds = model.predict(X_test)

output = pd.DataFrame({
    'PassengerId': test_data.PassengerId,
    'Survived': test_preds
})

output.to_csv('submission.csv', index=False)

print("Your submission was successfully saved!")

Validation accuracy: 0.8100558659217877
Your submission was successfully saved!


In [12]:
train_data["AgeGroup"] = pd.cut(
    train_data["Age"],
    bins=[0,10,20,30,40,50,60,100]
)

survival_by_age = train_data.groupby("AgeGroup")["Survived"].mean()

print(survival_by_age)

AgeGroup
(0, 10]      0.593750
(10, 20]     0.382609
(20, 30]     0.365217
(30, 40]     0.445161
(40, 50]     0.383721
(50, 60]     0.404762
(60, 100]    0.227273
Name: Survived, dtype: float64


/tmp/ipykernel_17/1448664975.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  survival_by_age = train_data.groupby("AgeGroup")["Survived"].mean()


In [13]:
female_subset = train_data[
    (train_data["Sex"] == "female") &
    (train_data["Age"] >= 10) &
    (train_data["Age"] <= 60)
]

survival_rate = female_subset["Survived"].mean()

print(survival_rate)

0.7675438596491229


In [14]:
train_data.groupby(["Sex", "AgeGroup"])["Survived"].mean()

/tmp/ipykernel_17/1213958645.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  train_data.groupby(["Sex", "AgeGroup"])["Survived"].mean()


Sex     AgeGroup 
female  (0, 10]      0.612903
        (10, 20]     0.739130
        (20, 30]     0.753086
        (30, 40]     0.836364
        (40, 50]     0.677419
        (50, 60]     0.928571
        (60, 100]    1.000000
male    (0, 10]      0.575758
        (10, 20]     0.144928
        (20, 30]     0.154362
        (30, 40]     0.230000
        (40, 50]     0.218182
        (50, 60]     0.142857
        (60, 100]    0.105263
Name: Survived, dtype: float64

In [15]:
train_data.groupby(["Pclass", "Sex"])["Survived"].mean()

Pclass  Sex   
1       female    0.968085
        male      0.368852
2       female    0.921053
        male      0.157407
3       female    0.500000
        male      0.135447
Name: Survived, dtype: float64

In [16]:
train_data.groupby(["Sex","AgeGroup"]).size()

/tmp/ipykernel_17/3387126171.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  train_data.groupby(["Sex","AgeGroup"]).size()


Sex     AgeGroup 
female  (0, 10]       31
        (10, 20]      46
        (20, 30]      81
        (30, 40]      55
        (40, 50]      31
        (50, 60]      14
        (60, 100]      3
male    (0, 10]       33
        (10, 20]      69
        (20, 30]     149
        (30, 40]     100
        (40, 50]      55
        (50, 60]      28
        (60, 100]     19
dtype: int64